# Data types & retrieval showcase — Swiss Legal & Financial AI (Phase 1)

This notebook makes **real, live calls** to each of the three Phase 1 ingestion sources
and shows, for every one of them:

1. the **raw** response as it comes off the wire (JSON payload, SPARQL bindings, or CSV text), and
2. the **cleaned / normalized** Python object our `src.ingestion.*` and `src.processing.*`
   modules turn it into.

No database or API key is required to run this — Fedlex, Curia Vista, and SNB are all
public, unauthenticated endpoints. Only network access is needed.

Run with the `Python (swiss-legal-ai uv)` kernel (installed via
`uv run python -m ipykernel install --user --name swiss-legal-ai --display-name "Python (swiss-legal-ai uv)"`
— see `VERIFY_SETUP.md`).

**Sources covered:**

| # | Source | Protocol | Client |
|---|--------|----------|--------|
| 1 | [Fedlex](https://fedlex.data.admin.ch) — Swiss federal law | SPARQL | `src.ingestion.fedlex_client.FedlexClient` |
| 2 | [Curia Vista](https://www.parlament.ch) — parliamentary business | OData v2 (JSON) | `src.ingestion.curia_vista_client.CuriaVistaClient` |
| 3 | [SNB Data Portal](https://data.snb.ch) — monetary/banking data | REST (CSV) + ETag | `src.ingestion.snb_client.SNBClient` |

> **opendata.swiss has been removed from this project.** It previously appeared here as a
> fourth source, used only for CKAN dataset *discovery* — it was never wired into any
> ingestion job (not called from `scripts/run_ingestion.py` or `src/api/main.py`), so
> deleting `opendata_swiss_client.py` (and its test) dropped no data from Qdrant or
> Postgres. See `swiss-legal-ai-architecture.md` §2.4 for the rationale.

> **Bugs fixed against the live endpoints** (rather than the mocked responses the test
> suite uses), each with a regression test:
> 1. `CuriaVistaClient.fetch_business` assumed `payload["d"]` could be a dict with a
>    `"results"` key; live responses return `payload["d"]` as a plain list directly, which
>    crashed on `.get()`.
> 2. `SNBClient.parse_cube_csv` assumed the CSV started at the header row; real exports are
>    prefixed with a BOM and a `CubeId` / `PublishingDate` metadata preamble, so every real
>    cube silently parsed to **zero** records.
> 3. `FedlexClient.fetch_act_html_text` did a plain GET on the act's ELI page
>    (`https://fedlex.data.admin.ch/eli/...`) itself — but that page is a JavaScript
>    single-page app, so the GET returned only a "please enable JavaScript" shell, never
>    the article text. Fixed by resolving the *actual* filestore document URI via SPARQL
>    first, then GETting that static file instead — see §1 below.
> 4. `CuriaVistaClient` never captured the `Language` field, never filtered by it, and never
>    stripped the HTML markup embedded in `InitialSituation`/`SubmittedText` — so a
>    `BusinessItem` could silently mix languages and carry raw `<p>`/`<h2>` tags into
>    chunking/embedding. Fixed: `fetch_business` now queries `Language eq 'IT'`, then
>    `'FR'`, then `'DE'` (project-wide language preference) and strips HTML before the text
>    is chunked — see §2 below.
>
> See `VERIFY_SETUP.md` for how to confirm these fixes with the test suite.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import json
from dataclasses import asdict
from datetime import date

import httpx
import pandas as pd

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Project root: C:\Users\Gabri\Documents\progetti\law_swiss


## 1. Fedlex — Swiss federal law (SPARQL)

Fedlex publishes consolidated (in-force) federal legislation as linked data using the
JOLux ontology, queryable via a public SPARQL endpoint. `FedlexClient` hand-writes two
query templates (see the module docstring for why: the ontology has a real learning
curve) rather than exposing a generic query builder.

We use systematic-number prefix `"642"` (direct federal tax law). `.env.example` ships
`FEDLEX_SYSTEMATIC_PREFIX=640`, but `"640"` itself is an unused/reserved SR chapter with
no in-force consolidated acts on the live endpoint (confirmed live: `640` → 0 acts, `641`
→ 5 acts, `642` → 5 acts) — worth knowing before treating that env default as "safe by
construction".

**Fetching article text is a two-step process** (§1c below): Fedlex's public ELI pages
(`https://fedlex.data.admin.ch/eli/...`) are a JavaScript single-page app — a plain GET
returns only a loading shell, never the text. The real text is a static HTML file in
Fedlex's public *filestore*, whose URI has to be resolved via a second SPARQL query
first. `FedlexClient.fetch_act_html_text` does this automatically.

In [2]:
from SPARQLWrapper import JSON, SPARQLWrapper

# --- 1a. The raw SPARQL response, straight off the wire -----------------------------
_raw_wrapper = SPARQLWrapper("https://fedlex.data.admin.ch/sparqlendpoint")
_raw_wrapper.setReturnFormat(JSON)
_raw_wrapper.setQuery(
    '''
    PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT DISTINCT ?consolidationAbstract ?srNotation ?title
    WHERE {
      ?consolidationAbstract a jolux:ConsolidationAbstract ;
                              jolux:classifiedByTaxonomyEntry ?taxEntry .
      ?taxEntry skos:notation ?srNotation .
      FILTER(STRSTARTS(STR(?srNotation), "642"))
      ?consolidation jolux:isMemberOf ?consolidationAbstract ;
                      jolux:isRealizedBy ?expression .
      ?expression jolux:title ?title ;
                  jolux:language <http://publications.europa.eu/resource/authority/language/DEU> .
    }
    LIMIT 3
    '''
)
raw_bindings = _raw_wrapper.query().convert()["results"]["bindings"]
print(f"type(raw_bindings) = {type(raw_bindings).__name__}, {len(raw_bindings)} rows\n")
print(json.dumps(raw_bindings[0], indent=2, ensure_ascii=False))

type(raw_bindings) = list, 3 rows

{
  "consolidationAbstract": {
    "type": "uri",
    "value": "https://fedlex.data.admin.ch/eli/cc/1990/1019_1019_1019"
  },
  "srNotation": {
    "type": "typed-literal",
    "datatype": "https://fedlex.data.admin.ch/vocabulary/notation-type/id-systematique",
    "value": "642.212"
  },
  "title": {
    "type": "literal",
    "value": "Consolidation: 642.212 - 1990-07-01"
  }
}


SPARQL's JSON results format wraps every scalar value in a `{"type": ..., "value": ...}`
object per binding — this is the shape `FedlexClient._run_query` receives and every
`row["field"]["value"]` access in the client unwraps.

In [3]:
from src.ingestion.fedlex_client import FedlexClient

fedlex = FedlexClient()

# --- 1b. Cleaned data: FedlexClient.list_consolidated_acts_by_prefix() -----------------
acts = fedlex.list_consolidated_acts_by_prefix("642", language="de", limit=5)
print(f"type(acts) = list[{type(acts[0]).__name__}], {len(acts)} items\n")
pd.DataFrame([asdict(a) for a in acts])

type(acts) = list[ActSummary], 5 items



,act_uri,systematic_number,title,language,valid_from,valid_to
0,https://fedlex.data.admin.ch/eli/cc/1993/2940_...,642.110.8,Consolidation: 642.110.8 - 2014-01-01,de,2014-01-01,None
1,https://fedlex.data.admin.ch/eli/cc/2010/111,642.114,Consolidation: 642.114 - 2011-01-01,de,2011-01-01,None
2,https://fedlex.data.admin.ch/eli/cc/2012/449,642.115.325.1,Consolidation: 642.115.325.1 - 2021-01-01,de,2021-01-01,None
3,https://fedlex.data.admin.ch/eli/cc/2018/212,642.116,Consolidation: 642.116 - 2020-01-01,de,2020-01-01,None
4,https://fedlex.data.admin.ch/eli/cc/1992/1795_...,642.116.1,Consolidation: 642.116.1 - 1995-01-01,de,1995-01-01,None


`ActSummary` is a plain dataclass — five typed fields extracted from the nested SPARQL
binding shape above. Note `title` here is Fedlex's auto-generated consolidation label
(e.g. `"Consolidation: 642.110.8 - 2014-01-01"`), not the act's actual legal name — that
requires a second query (`fetch_act`, below) against the expression's `jolux:title`.

In [4]:
# --- 1c. Two-step text fetch: resolve the filestore file URI via SPARQL, then GET it ---
# Step 1: SPARQL resolves the *actual* document URI (Fedlex filestore, static HTML) —
# NOT the act's ELI page, which is a JavaScript app and returns no text on a plain GET.
file_url = fedlex.fetch_act_file_url(acts[0].act_uri, language="de")
print("ELI page (act_uri, JS app — do NOT GET this for text):")
print(" ", acts[0].act_uri)
print("Filestore file URL (static HTML, resolved via SPARQL — GET this instead):")
print(" ", file_url)

# Step 2: fetch_act(..., include_text=True) runs both steps above and parses the result
# with BeautifulSoup.
act = fedlex.fetch_act(acts[0].act_uri, language="de", include_text=True)
print("\nActDocument fields (excluding raw_text):")
for k, v in asdict(act).items():
    if k not in ("raw_text", "articles"):
        print(f"  {k}: {v}")
print(f"\nraw_text: {len(act.raw_text)} chars, starts with:\n{act.raw_text[:300]!r}")

ELI page (act_uri, JS app — do NOT GET this for text):
  https://fedlex.data.admin.ch/eli/cc/1993/2940_2940_2940
Filestore file URL (static HTML, resolved via SPARQL — GET this instead):
  https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/cc/1993/2940_2940_2940/20140101/de/html/fedlex-data-admin-ch-eli-cc-1993-2940_2940_2940-20140101-de-html-7.html



ActDocument fields (excluding raw_text):
  act_uri: https://fedlex.data.admin.ch/eli/cc/1993/2940_2940_2940
  systematic_number: 642.110.8
  title: Verordnung vom 20. Oktober 1993 über die Besteuerung von natürlichen Personen im Ausland mit einem Arbeitsverhältnis zum Bund oder zu einer andern öffentlichrechtlichen Körperschaft oder Anstalt des Inlandes
  language: de
  valid_from: None
  valid_to: None
  source_url: https://fedlex.data.admin.ch/eli/cc/1993/2940_2940_2940

raw_text: 5948 chars, starts with:
'\n\ninput-de\n642.110.8 \nVerordnung\nüber die Besteuerung von natürlichen Personen \nim Ausland mit einem Arbeitsverhältnis zum Bund \noder zu einer andern öffentlichrechtlichen Körperschaft \noder Anstalt des Inlandes\nvom 20. Oktober 1993 (Stand am 1. Januar 2014)\nDer Schweizerische Bundesrat,\ngestützt au'


**Fixed, verified live**: the snippet above no longer starts with a "please enable
JavaScript" notice — `raw_text` now contains the actual consolidated act, because
`fetch_act_html_text` GETs the filestore file URL resolved by SPARQL
(`fetch_act_file_url`), not the act's JS-rendered ELI page. The chunking step below
confirms this further: it finds one chunk per real `Art. N` header rather than a
single preamble-only chunk.

In [5]:
from src.processing.chunker import chunk_act_text
from src.processing.metadata import build_metadata
from src.processing.normalizer import clean_text, detect_language

# --- 1d. Cleaned data, continued: normalize -> chunk -> metadata ----------------------
cleaned = clean_text(act.raw_text)
print("detected language:", detect_language(cleaned))

chunks = chunk_act_text(act.raw_text)
print(f"\ntype(chunks) = list[Chunk], {len(chunks)} chunk(s)")
for c in chunks:
    print(f"  Chunk(article={c.article!r}, paragraph_index={c.paragraph_index}, len(text)={len(c.text)})")

metadata = build_metadata(
    chunks[0],
    source="fedlex",
    language="de",
    source_url=act.source_url,
    systematic_number=act.systematic_number,
    valid_from=act.valid_from,
    valid_to=act.valid_to,
)
print("\nChunkMetadata (pydantic model) for chunk 0:")
metadata.model_dump()

detected language: de

type(chunks) = list[Chunk], 11 chunk(s)
  Chunk(article=None, paragraph_index=None, len(text)=506)
  Chunk(article='1', paragraph_index=None, len(text)=748)
  Chunk(article='2', paragraph_index=None, len(text)=932)
  Chunk(article='3', paragraph_index=None, len(text)=254)
  Chunk(article='4', paragraph_index=None, len(text)=393)
  Chunk(article='5', paragraph_index=None, len(text)=1259)
  Chunk(article='6', paragraph_index=None, len(text)=209)
  Chunk(article='7', paragraph_index=None, len(text)=675)
  Chunk(article='8', paragraph_index=None, len(text)=402)
  Chunk(article='9', paragraph_index=None, len(text)=376)
  Chunk(article='10', paragraph_index=None, len(text)=159)

ChunkMetadata (pydantic model) for chunk 0:


{'source': 'fedlex',
 'level': 'federal',
 'law_short_name': None,
 'article': None,
 'systematic_number': '642.110.8',
 'language': 'de',
 'valid_from': None,
 'valid_to': None,
 'source_url': 'https://fedlex.data.admin.ch/eli/cc/1993/2940_2940_2940',
 'ingested_at': '2026-07-27T13:36:17.841619+00:00',
 'content_hash': '0a46950466990efe13dca5c8a1858ae44f10db840afefe330617bd813828206d'}

## 2. Curia Vista — parliamentary business (OData v2)

`ws.parlament.ch` exposes an OData v2 service. `CuriaVistaClient` hand-builds the
`$filter`/`$top`/`$format` query string (an OData library was judged unnecessary — see
the module docstring) and normalizes each `Business` entity into a `BusinessItem`.

Each `Business` row is keyed by `(ID, Language)` — the same business item comes back once
per language, each with its own localized `Title`/`InitialSituation`. `fetch_business`
queries `Language eq 'IT'`, then `'FR'`, then `'DE'`, and merges by ID — so every item is
returned in Italian when available, falling back to French then German (the project-wide
language preference).


In [6]:
from src.ingestion.curia_vista_client import CURIA_VISTA_BASE_URL

# --- 2a. The raw OData JSON response, filtered to Italian, for a recent month ----------
_params = {
    "$filter": (
        "SubmissionDate ge datetime'2026-06-01T00:00:00' "
        "and SubmissionDate le datetime'2026-06-30T00:00:00' "
        "and Language eq 'IT'"
    ),
    "$top": "2",
    "$format": "json",
}
_resp = httpx.get(f"{CURIA_VISTA_BASE_URL}/Business", params=_params, timeout=20, follow_redirects=True)
_raw_payload = _resp.json()
print("status:", _resp.status_code)
print(f'type(payload["d"]) = {type(_raw_payload["d"]).__name__}  <- a plain list on the live API,')
print("not {'results': [...]} as the older OData verbose-JSON convention would suggest.\n")

_raw_entry = _raw_payload["d"][0]
print("Language:", _raw_entry["Language"], "| BusinessShortNumber:", _raw_entry["BusinessShortNumber"])
print("\nInitialSituation (raw, HTML-encoded) — first 400 chars:")
print(_raw_entry["InitialSituation"][:400])


status: 200
type(payload["d"]) = list  <- a plain list on the live API,
not {'results': [...]} as the older OData verbose-JSON convention would suggest.

Language: IT | BusinessShortNumber: 26.046

InitialSituation (raw, HTML-encoded) — first 400 chars:
<h2 class="Titel_d"><strong>Comunicato stampa del Consiglio federale del 05.06.2026</strong></h2><p class="Standard_d"><strong>Il Consiglio federale adotta il messaggio sull’impiego di vaccini contro le epizoozie nelle situazioni di emergenza</strong></p><p class="Standard_d">&nbsp;Il 5 giugno 2026 il Consiglio federale ha adottato il messaggio concernente la modifica della legge sulle epizoozie. 


In [7]:
from src.ingestion.curia_vista_client import CuriaVistaClient


async def _fetch_curia_vista():
    async with CuriaVistaClient() as client:
        return await client.fetch_business(date(2026, 6, 1), date(2026, 6, 30), top=5)


# --- 2b. Cleaned data: CuriaVistaClient.fetch_business() -------------------------------
# Note this makes 3 requests under the hood (Language eq 'IT'/'FR'/'DE') and merges them.
business_items = await _fetch_curia_vista()
print(f"type(business_items) = list[{type(business_items[0]).__name__}], {len(business_items)} items\n")
print("languages present:", sorted({b.language for b in business_items}))
assert all("<" not in (b.summary or "") for b in business_items), "HTML should be stripped by now"

df_business = pd.DataFrame([asdict(b) for b in business_items])
df_business["summary"] = df_business["summary"].str.slice(0, 80) + "…"
df_business


2026-07-27 15:36:17,989 INFO swiss_legal_ai.ingestion GET /Business


2026-07-27 15:36:18,496 INFO swiss_legal_ai.ingestion GET /Business


2026-07-27 15:36:18,968 INFO swiss_legal_ai.ingestion GET /Business


2026-07-27 15:36:19,485 INFO swiss_legal_ai.ingestion.curia_vista Fetched 5 distinct Curia Vista business items for 2026-06-01..2026-06-30 (languages=('IT', 'FR', 'DE'))


type(business_items) = list[BusinessItem], 5 items

languages present: ['it']


,id,short_number,title,summary,status,submission_date,business_type,language,related_law_references,source_url
0,20260046,26.046,Legge sulle epizoozie. Modifica,Comunicato stampa del Consiglio federale del 0...,Nella Commissione del Consiglio degli Stati,2026-06-05,Oggetto del Consiglio federale,it,[],https://www.parlament.ch/it/ratsbetrieb/suche-...
1,20260047,26.047,Convenzioni dell’Organizzazione internazionale...,Comunicato stampa del Consiglio federale del 0...,Nella Commissione del Consiglio degli Stati,2026-06-05,Oggetto del Consiglio federale,it,[],https://www.parlament.ch/it/ratsbetrieb/suche-...
2,20260048,26.048,Sviluppo dell’acquis di Schengen. Approvazione,Comunicato stampa del Consiglio federale del 0...,Nella Commissione del Consiglio degli Stati,2026-06-05,Oggetto del Consiglio federale,it,[],https://www.parlament.ch/it/ratsbetrieb/suche-...
3,20260052,26.052,Finanziamento delle centrali elettriche di ris...,Comunicato stampa del Consiglio federale del 1...,Assegnato alla commissione competente,2026-06-12,Oggetto del Consiglio federale,it,[],https://www.parlament.ch/it/ratsbetrieb/suche-...
4,20260053,26.053,Crediti d’impegno per manifestazioni sportive ...,Comunicato stampa del Consiglio federale del 0...,Nella Commissione del Consiglio nazionale,2026-06-05,Oggetto del Consiglio federale,it,[],https://www.parlament.ch/it/ratsbetrieb/suche-...


`BusinessItem.related_law_references` is populated by a regex heuristic
(`_extract_law_references`) that scans the title + summary text for `SR ###.##`-style
citations — a light heuristic, not a legal citation parser (per the module docstring),
useful for flagging items worth cross-linking to Fedlex acts during retrieval.

### 2c. Metadata coherence with Fedlex, and the Qdrant/Postgres split

Curia Vista uses the same `ChunkMetadata` schema as Fedlex (architecture spec §3.2) —
`source` differs (`"curia_vista"` vs `"fedlex"`), but `language`, `source_url`, and
`valid_from` line up, and `article` is repurposed to hold the business short number
(`26.046`) — the closest Curia Vista analogue to a Fedlex article number, and what a
citation would reference. This is what lets hybrid search in Qdrant filter across both
sources with the same field names.

Per the architecture spec, each business item is **dual-written**: the single synthetic
chunk below is embedded into Qdrant (semantic search), while the *full* raw fields
(all of `BusinessItem`) go into PostgreSQL's `curia_vista_business` table (exact lookup) —
see `scripts/run_ingestion.py::ingest_curia_vista`.


In [8]:
from src.processing.chunker import chunk_business_item

# --- 2c. One synthetic chunk per business item, same metadata schema as Fedlex --------
_item = business_items[0]
cv_chunk = chunk_business_item(_item.short_number, _item.title, _item.status, _item.summary)
cv_metadata = build_metadata(
    cv_chunk,
    source="curia_vista",
    language=_item.language,
    source_url=_item.source_url,
    law_short_name=_item.business_type,
    valid_from=_item.submission_date,
)
print("Chunk.text:\n", cv_chunk.text[:400], "\n")
cv_metadata.model_dump()


Chunk.text:
 26.046 — Legge sulle epizoozie. Modifica

Status: Nella Commissione del Consiglio degli Stati

Comunicato stampa del Consiglio federale del 05.06.2026 Il Consiglio federale adotta il messaggio sull’impiego di vaccini contro le epizoozie nelle situazioni di emergenza Il 5 giugno 2026 il Consiglio federale ha adottato il messaggio concernente la modifica della legge sulle epizoozie. L’obiettivo è cr 



{'source': 'curia_vista',
 'level': 'federal',
 'law_short_name': 'Oggetto del Consiglio federale',
 'article': '26.046',
 'systematic_number': None,
 'language': 'it',
 'valid_from': '2026-06-05',
 'valid_to': None,
 'source_url': 'https://www.parlament.ch/it/ratsbetrieb/suche-curia-vista/geschaeft?AffairId=20260046',
 'ingested_at': '2026-07-27T13:36:19.498008+00:00',
 'content_hash': 'dda4680607ce4218485084fb12b518b3d7d2366f5ea5edb77df5156dc7478b04'}

## 3. SNB Data Portal — monetary & banking data (REST + ETag)

The SNB asks integrators to only re-fetch a cube when it has actually changed. `SNBClient`
always sends the previously-stored ETag as `If-None-Match` and treats HTTP 304 as
"nothing to do" — demonstrated in 3d below with two real, back-to-back requests.

This is the one source whose data must **never** reach Qdrant: it's numeric time series,
queried exactly via SQL (PostgreSQL), not semantically searched.


In [9]:
from src.config import get_settings
import inspect
from src.ingestion.snb_client import SNBClient

_settings = get_settings()
print("Configured SNB_TEST_CUBES (settings.snb_test_cubes):", _settings.snb_test_cubes)

# --- Destination check: SNB numeric time series must go to Postgres, never Qdrant ------
_snb_source = inspect.getsource(SNBClient)
assert "qdrant" not in _snb_source.lower(), "SNBClient must not import/touch Qdrant"
print("Confirmed: SNBClient has no Qdrant dependency — sync_cube() only writes to PostgresStore.")


Configured SNB_TEST_CUBES (settings.snb_test_cubes): rendoblim,snbmonagg
Confirmed: SNBClient has no Qdrant dependency — sync_cube() only writes to PostgresStore.


In [10]:
from src.ingestion.snb_client import SNB_BASE_URL

# --- 4a. The raw CSV response -----------------------------------------------------------
_resp = httpx.get(f"{SNB_BASE_URL}/api/cube/rendoblim/data/csv/en", timeout=20, follow_redirects=True)
_raw_csv = _resp.text
print("status:", _resp.status_code, "| ETag:", _resp.headers.get("ETag"))
print(f"content length: {len(_raw_csv)} chars\n")
print("first 6 lines (note the BOM + CubeId/PublishingDate preamble before the real header row):")
for line in _raw_csv.splitlines()[:6]:
    print(" ", repr(line))

status: 200 | ETag: "092c612ae5915c8d593257dc02b332a205c2de81457ede4e580a8e8f2fe299bd8"
content length: 142136 chars

first 6 lines (note the BOM + CubeId/PublishingDate preamble before the real header row):
  '\ufeff"CubeId";"rendoblim"'
  '"PublishingDate";"2025-09-01 14:29"'
  ''
  '"Date";"D0";"Value"'
  '"1988-01";"1J";"2.887"'
  '"1988-01";"2J";"3.218"'


In [11]:
from src.ingestion.snb_client import SNBClient

# --- 4b. Cleaned data: SNBClient.parse_cube_csv() ---------------------------------------
records = SNBClient.parse_cube_csv(_raw_csv)
print(f"type(records) = list[dict], {len(records)} records\n")
df_snb = pd.DataFrame(records)
df_snb.head()

type(records) = list[dict], 5744 records



,dimensions,observation_date,value
0,{'D0': '1J'},1988-01,2.887
1,{'D0': '2J'},1988-01,3.218
2,{'D0': '3J'},1988-01,3.393
3,{'D0': '4J'},1988-01,3.554
4,{'D0': '5J'},1988-01,3.695


In [12]:
async def _fetch_snb_twice():
    async with SNBClient() as client:
        text1, etag1 = await client.fetch_cube_csv("rendoblim", lang="en")
        text2, etag2 = await client.fetch_cube_csv("rendoblim", lang="en", etag=etag1)
        return (text1 is not None, etag1), (text2, etag2)


# --- 4c. ETag-based conditional GET, live -----------------------------------------------
(first_had_body, etag), (second_body, etag2) = await _fetch_snb_twice()
print(f"1st call (no etag sent):        got a body = {first_had_body}, ETag = {etag}")
print(f"2nd call (etag sent as If-None-Match): got a body = {second_body is not None} (None means HTTP 304 Not Modified)")
assert etag == etag2, 'ETag should be stable across the two calls'

2026-07-27 15:36:19,845 INFO swiss_legal_ai.ingestion GET /api/cube/rendoblim/data/csv/en


2026-07-27 15:36:20,060 INFO swiss_legal_ai.ingestion GET /api/cube/rendoblim/data/csv/en


2026-07-27 15:36:20,189 INFO swiss_legal_ai.ingestion.snb Cube rendoblim not modified since last fetch (etag="092c612ae5915c8d593257dc02b332a205c2de81457ede4e580a8e8f2fe299bd8")


1st call (no etag sent):        got a body = True, ETag = "092c612ae5915c8d593257dc02b332a205c2de81457ede4e580a8e8f2fe299bd8"
2nd call (etag sent as If-None-Match): got a body = False (None means HTTP 304 Not Modified)


## Summary: data types at a glance

| Stage | Type | Where |
|---|---|---|
| Fedlex raw | SPARQL JSON bindings (`dict`) | wire format |
| Fedlex cleaned | `ActSummary`, `ActDocument`, `Article` (dataclasses) | `src.ingestion.fedlex_client` |
| Curia Vista raw | OData JSON (`{"d": [...]}, list of dict`, one row per `(ID, Language)`) | wire format |
| Curia Vista cleaned | `BusinessItem` (dataclass, HTML-stripped, IT→FR→DE language priority) | `src.ingestion.curia_vista_client` |
| SNB raw | CSV text (`str`, `;`-delimited, BOM + metadata preamble) | wire format |
| SNB cleaned | `list[dict]` — `{"dimensions", "observation_date", "value"}` | `src.ingestion.snb_client` |
| Chunking (Fedlex) | `Chunk` — one per article (or paragraph group) | `src.processing.chunker.chunk_act_text` |
| Chunking (Curia Vista) | `Chunk` — one synthetic summary chunk per business item | `src.processing.chunker.chunk_business_item` |
| Metadata | `ChunkMetadata` (pydantic `BaseModel`), same schema for both text sources | `src.processing.metadata` |

**Destinations** (architecture spec §3.3):
- **Qdrant** (embedded, semantically searched): Fedlex article chunks + Curia Vista
  business-item chunks, sharing one `ChunkMetadata` schema so hybrid search can filter
  across sources by `source`, `language`, `valid_to`, etc.
- **PostgreSQL** (structured, exact lookup only, never embedded): Curia Vista's full raw
  fields (all of `BusinessItem`) and every SNB numeric observation. SNB data never reaches
  Qdrant — see the destination check in §3 above.

opendata.swiss has been removed from the project entirely (see the note at the top of
this notebook and `swiss-legal-ai-architecture.md` §2.4).

Everything past this point in the real pipeline (embedding → Qdrant/Postgres storage →
hybrid retrieval) requires the Docker services from `docker compose up -d` — see
`scripts/run_ingestion.py` for the end-to-end job (now including Curia Vista's Qdrant
upsert) and `VERIFY_SETUP.md` for how to check that path works.


In [13]:
fedlex.close()  # no-op for FedlexClient, included for symmetry with the async clients' __aexit__